# 11.15 中文的小筆畫，能靠放大模型救回來嗎？


把紙上的「未」與「末」縮到很小，兩條橫線的差別可能糊在一起。模型收到的是縮完的數字，並沒有偷偷保留原紙張。這像先把文件影印得模糊，再請更聰明的人讀：他可能根據句子猜字，卻不能保證把已消失的筆畫找回來。

前置是[11.9縮圖與裁切](https://birdhackor.github.io/tiny-perceptron-vlm/11.9.html)與[11.12讀字任務](https://birdhackor.github.io/tiny-perceptron-vlm/11.12.html)。下面不畫完整中文字，只留一条一像素寬的亮線，隔離「縮小會如何影響細節」這件事。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/practical_stroke.svg")))

In [ ]:
from tiny_perceptron.natural_concepts import thin_stroke_report

report = thin_stroke_report()
print("原圖與縮圖尺寸", report["original_shape"], report["small_shape"])
print("最亮的數字", report["original_brightest"], report["small_brightest"])
print("至少半亮的數字個數", report["pixels_at_least_half_bright_before"], report["pixels_at_least_half_bright_after"])

`thin_stroke_report()`先建立一張32乘32的黑圖，把其中一整欄的32個數字設成1，再把每個8乘8區域平均成一個數字。它把比較結果放進名為`report`的字典；`report["original_shape"]`這種寫法，是按項目名稱取出原圖尺寸，其他名稱分別取縮圖尺寸、最亮數字與達到門檻的個數。

原圖32乘32縮成4乘4，最亮數字從1變成0.125；至少0.5的數字從32個變成0個。這次用區域平均：一格裡八欄只有一欄亮，因此只留下八分之一。門檻0.5只是讓變化容易看見，不是OCR模型必然使用的規則；這個實驗沒有測辨字能力。

實際做法可以保留較高解析度，或把一張大圖切成多個小窗，再保留各窗在原圖的位置。像先看文件全貌，再把小字區域放大讀，兩種線索都有用途。小窗越多，模型要處理的圖片位置也通常越多，時間和記憶體隨之增加；不能把放大圖片當成免費的能力。圖片本來就模糊、字被遮住時，仍應允許回答「看不清」。

[第20章](https://birdhackor.github.io/tiny-perceptron-vlm/20.1.html)的實用成品接手已學過大量圖文的模型；這份起始權重叫作底座。它配套的圖片處理器，負責把原圖轉成模型接收的尺寸與區塊。我們再用本課示範回答調整部分權重，這一步叫微調。前面的16乘16數字OCR只練過數字，不足以證明能閱讀中文。

練習說明：把生成回答的文字模型放大、保留原圖細節、補上中文字圖片及正確文字示範，分別解決哪一種問題？先想想增加可學的數字能否找回已縮掉的筆畫，再想想模型如何得知某個可見字形應回答哪個字。
